In [ ]:
#%matplotlib widget
import json
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import io, h5py, zipfile
import numpy as np
from tqdm import tqdm
import os
import mienc.support as ms
from mienc import Corrector, NonLinearEstimator
import pandas as pd
from scipy.signal import hilbert
from scipy.io import loadmat, savemat
from scipy.stats import ks_1samp, uniform, pearsonr, spearmanr
import seaborn as sns

In [ ]:
wtf = np.load("../NonLinearityResultsNew/localised/fMRI_AAL90_region_quantiles_strin.npy")
wtf.shape

In [ ]:
def summarise_localised_non_linearity(results_file, subset_description, subset_identifiers, subset_names, output_prefix, pair_num, subj_num, elec_num, region_positions, quantiles, nbins, nsteps):
    corrct = Corrector(nbins, nsteps, folder_name=os.path.dirname(results_file).format(subset_identifiers[0]), config="config.ini")
    corrct.compute_correction()
    for subset_id, subset_na in zip(subset_identifiers, subset_names):
        region_quantiles=np.zeros((subj_num, pair_num))
        if os.path.isfile(f"../NonLinearityResultsNew/localised/{output_prefix}_region_quantiles_{subset_id}.npy"):
            region_quantiles = np.load(f"../NonLinearityResultsNew/localised/{output_prefix}_region_quantiles_{subset_id}.npy")
            print(f"Region quantiles for {subset_description} {subset_id} loaded")
        else:
            for s in tqdm(range(subj_num), "Subject"):
                pat = np.load(results_file.format(subset_id, s))
                true = pat[:,0]
                surr = np.sort(pat[:,1:], 1)
                for i in range(pair_num):
                    region_quantiles[s,i] = np.searchsorted(surr[i,:], true[i], "left")
                np.save(f"../NonLinearityResultsNew/localised/{output_prefix}_region_quantiles_{subset_id}.npy", region_quantiles)
        
        fig, ax = plt.subplots(1,2, gridspec_kw={'width_ratios': [2, 3]}, figsize=(8,5))
        plt.sca(ax[0])
        plt.hist(region_quantiles[:,0], bins=np.arange(0,110,10))
        plt.ylabel("# pairs")
        plt.xlabel("%ile")

        plt.sca(ax[0].twinx())
        who, counts = np.unique(region_quantiles[:,0]+1, return_counts=True)
        truecum = np.cumsum(counts)/subj_num
        exp = np.arange(0,101)/100
        plt.plot(who, truecum, color="black")
        plt.plot(exp, color="orange")
        diff = exp[who.astype(int)]-truecum
        ks_stat = np.max(diff)
        where = np.argmax(diff)
        plt.vlines(who[where], truecum[where], exp[int(who[where])],"red")
        plt.title(f"K-S statistic: {ks_stat:.3} at {who[where]}")
        plt.xlim((0,100))
        plt.ylim((0,1))
        plt.ylabel("cumulative")
        plt.title(subset_description+subset_na+" - Sample K-S plot")

        if os.path.isfile(f"../NonLinearityResultsNew/localised/{output_prefix}_ks_stat_{subset_id}.npy") and os.path.isfile(f"../NonLinearityResultsNew/localised/{output_prefix}_ks_p_{subset_id}.npy"):
            ks_stat = np.load(f"../NonLinearityResultsNew/localised/{output_prefix}_ks_stat_{subset_id}.npy")
            ks_p = np.load(f"../NonLinearityResultsNew/localised/{output_prefix}_ks_p_{subset_id}.npy")
            print(f"K-S stats for {subset_description} {subset_id} loaded")
        else:
            ks_stat = np.zeros(pair_num)
            ks_p = np.zeros(pair_num)
            exp = np.arange(0,101)/100
            for i in tqdm(range(pair_num), "Pair"):
                stat, pval = ks_1samp(region_quantiles[:,i], uniform.cdf,(0,100), alternative="less")
                ks_stat[i] = stat
                ks_p[i] = pval
            np.save(f"../NonLinearityResultsNew/localised/{output_prefix}_ks_stat_{subset_id}.npy", ks_stat)
            np.save(f"../NonLinearityResultsNew/localised/{output_prefix}_ks_p_{subset_id}.npy", ks_p)

        plt.sca(ax[1])
        sorted_p = np.sort(ks_p.flatten())[::-1]
        plt.plot(sorted_p)
        plt.plot(np.arange(pair_num),0.05/(pair_num-np.arange(pair_num)),":r")
        plt.plot([0,pair_num-1],[0.05/pair_num,0.05/pair_num], ":k")
        plt.yscale("log")

        distance_from_thresh = sorted_p - 0.05/(pair_num-np.arange(pair_num))
        distance_from_thresh[distance_from_thresh>0]=-np.inf
        which = np.argmax(distance_from_thresh)
        ax[1].yaxis.tick_right()
        ax[1].yaxis.set_label_position("right")
        plt.ylabel("$p$-value")
        plt.xlabel("pair")
        plt.title(f"Holm-Bonferroni threshold: {sorted_p[which]:.4}")
        plt.show()

        corrected = np.full(pair_num, np.nan)
        thresh = sorted_p[which] # holm-bonferroni 1-(1-0.05)**(1/pair_num)# šidák 0.05/pair_num # bonferroni
        corrected[ks_p<thresh] = ks_stat[ks_p<thresh]
        mappa = np.zeros([elec_num,elec_num])
        mappa[np.triu_indices(elec_num,1)]=corrected
        mappa+=mappa.T
        siginreg = np.sum(mappa>0, 1)
        print("Non linear connections:", np.sum(siginreg)/2)
        fix, ax = plt.subplots(1,2, gridspec_kw={'width_ratios': [1.5,2]}, figsize=(12,5))
        plt.sca(ax[0])
        plt.imshow(mappa[siginreg>0,:][:,siginreg>0], interpolation="none")
        plt.colorbar(shrink=0.7).ax.set_ylabel('KS statistcs', rotation=90)
        plt.yticks(np.arange(0,sum(siginreg>0),10), np.arange(elec_num)[siginreg>0][::10])
        step = 10 if len(np.arange(0,sum(siginreg>0),10))<12 else 20
        plt.xticks(np.arange(0,sum(siginreg>0),step), np.arange(elec_num)[siginreg>0][::step])
        plt.xlabel(f"{sum(siginreg>0)}/{elec_num} electrodes")

        plt.sca(ax[1])
        plt.plot(np.sum(mappa>0, 1), "o")
        plt.xticks(np.arange(0,elec_num,10), rotation=45)
        plt.xlabel(f"Electrode")
        plt.ylabel("# significant non-linear connections")
        plt.suptitle(subset_description+subset_na+f" - {np.sum(ks_p<thresh)} ({100*np.sum(ks_p<thresh)/pair_num:.3} %) significant pairs")
        plt.show()

        if region_positions is not None:
            fig, ax = plt.subplots(1,2, gridspec_kw={'width_ratios': [2,1]},figsize=(8,6))
            ax1 = fig.add_subplot(121,projection='3d')
            ax[0].axis("off")
            linear = region_positions[siginreg==0]
            lt, mt = np.quantile(siginreg[siginreg>0], quantiles)
            low = region_positions[np.logical_and(siginreg>0, siginreg<=lt)]
            mild = region_positions[np.logical_and(siginreg>lt, siginreg<=mt)]
            very = region_positions[siginreg>mt]

            ax1.scatter(linear.x, linear.y, linear.z, marker="o", color="gray")
            ax1.scatter(low.x, low.y, low.z, marker="o", color="yellow")
            ax1.scatter(mild.x, mild.y, mild.z, marker="o", color="orange")
            ax1.scatter(very.x, very.y, very.z, marker="o", color="red")

            ax1.set_xlabel('Frontal')
            ax1.set_ylabel('Anteroposterior')
            ax1.set_zlabel('Craniocaudal')
            plt.sca(ax[1])
            legend_elements = [Line2D([0], [0], marker='o', color='w', label='0', markerfacecolor='gray', markersize=15),
                            Line2D([0], [0], marker='o', color='w', label=f'1 - {lt:.3} ({int(quantiles[0]*100)} %ile)', markerfacecolor='yellow', markersize=15),
                            Line2D([0], [0], marker='o', color='w', label=f'{lt:.3} - {mt:.3} ({int(quantiles[1]*100)} %ile)', markerfacecolor='orange', markersize=15),
                            Line2D([0], [0], marker='o', color='w', label=f'> {mt:.3}', markerfacecolor='red', markersize=15)]
            ax[1].legend(handles=legend_elements, loc='center', title="Number of significantly\nnon linear connections", frameon=False)
            ax[1].axis("off")
            plt.suptitle(subset_description+subset_na+" - Non-Linearity position")
            plt.show()

        if os.path.isfile(f"../NonLinearityResultsNew/localised/{output_prefix}_all_regions_quantiles_{subset_id}.npy") and os.path.isfile(f"../NonLinearityResultsNew/localised/{output_prefix}_all_regions_MI_{subset_id}.npy"):
            all_regions_quantiles = np.load(f"../NonLinearityResultsNew/localised/{output_prefix}_all_regions_quantiles_{subset_id}.npy")
            all_regions_MI = np.load(f"../NonLinearityResultsNew/localised/{output_prefix}_all_regions_MI_{subset_id}.npy")
            print(f"All Regions quantiles for {subset_description} {subset_id} loaded")
        else:
            all_of_it = np.zeros((elec_num, elec_num, subj_num, 100))
            for i in tqdm(range(subj_num), desc="Load and correct"):
                all_of_it[(*np.triu_indices(elec_num,1),i,slice(None))] = corrct.correct(np.load(results_file.format(subset_id, i)))
            all_of_it+=all_of_it.transpose((1,0,2,3))
            all_regions = np.sum(all_of_it, axis=0)/(elec_num-1)

            all_regions_quantiles = np.zeros((subj_num, elec_num))
            for s in tqdm(range(subj_num), desc="Quantiles subj"):
                true = all_regions[:,s,0]
                surr = np.sort(all_regions[:,s,1:], 1)
                for i in range(elec_num):
                    all_regions_quantiles[s,i] = np.searchsorted(surr[i,:], true[i], "left")
            np.save(f"../NonLinearityResultsNew/localised/{output_prefix}_all_regions_quantiles_{subset_id}.npy", all_regions_quantiles)
            all_regions_MI = np.zeros((elec_num, subj_num),[("TMI","f8"),("GMI","f8"),("sGMI","f8")])
            all_regions_MI["TMI"]=all_regions[:,:,0]
            all_regions_MI["GMI"]=np.mean(all_regions[:,:,1:], axis=-1)
            all_regions_MI["sGMI"]=np.std(all_regions[:,:,1:], axis=-1)
            np.save(f"../NonLinearityResultsNew/localised/{output_prefix}_all_regions_MI_{subset_id}.npy", all_regions_MI)

        fig, ax = plt.subplots(1,2, gridspec_kw={'width_ratios': [2, 3]}, figsize=(8,5))
        plt.sca(ax[0])
        plt.hist(all_regions_quantiles[:,0], bins=np.arange(0,110,10))
        plt.ylabel("# pairs")
        plt.xlabel("%ile")

        plt.sca(ax[0].twinx())
        who, counts = np.unique(all_regions_quantiles[:,0]+1, return_counts=True)
        truecum = np.cumsum(counts)/subj_num
        exp = np.arange(0,101)/100
        plt.plot(who, truecum, color="black")
        plt.plot(exp, color="orange")
        diff = exp[who.astype(int)]-truecum
        ks_stat = np.max(diff)
        where = np.argmax(diff)
        plt.vlines(who[where], truecum[where], exp[int(who[where])],"red")
        plt.title(f"K-S statistic: {ks_stat:.3} at {who[where]}")
        plt.xlim((0,100))
        plt.ylim((0,1))
        plt.ylabel("cumulative")
        plt.title(subset_description+subset_na+" - Sample K-S plot - regions")
        
        regions_ks_stat = np.zeros(elec_num)
        regions_ks_p = np.zeros(elec_num)
        exp = np.arange(0,101)/100
        for i in range(elec_num):
            stat, pval = ks_1samp(all_regions_quantiles[:,i], uniform.cdf,(0,100), alternative="less")
            regions_ks_stat[i] = stat
            regions_ks_p[i] = pval
        sorted_p = np.sort(regions_ks_p.flatten())[::-1]

        distance_from_thresh = sorted_p - 0.05/(elec_num-np.arange(elec_num))
        distance_from_thresh[distance_from_thresh>0]=-np.inf
        which = np.argmax(distance_from_thresh)
        corrected = np.full(elec_num, np.nan)
        thresh = sorted_p[which] # holm-bonferroni 1-(1-0.05)**(1/pair_num)# šidák 0.05/pair_num # bonferroni
        corrected[regions_ks_p<thresh] = regions_ks_stat[regions_ks_p<thresh]

        plt.sca(ax[1])
        plt.plot(sorted_p)
        plt.plot(np.arange(elec_num),0.05/(elec_num-np.arange(elec_num)),":r")
        plt.plot([0,elec_num-1],[0.05/elec_num,0.05/elec_num], ":k")
        plt.yscale("log")
        ax[1].yaxis.tick_right()
        ax[1].yaxis.set_label_position("right")
        plt.ylabel("$p$-value")
        plt.xlabel("pair")
        plt.title(f"Holm-Bonferroni threshold: {sorted_p[which]:.4} - region")
        plt.show()

        if region_positions is not None:
            fig, ax = plt.subplots(1,2, gridspec_kw={'width_ratios': [8,1]},figsize=(8,6))
            ax1 = fig.add_subplot(121,projection='3d')
            ax[0].axis("off")
            
            ax1.scatter(region_positions[np.isnan(corrected)].x, region_positions[np.isnan(corrected)].y, region_positions[np.isnan(corrected)].z, marker="o", c="grey")
            scat = ax1.scatter(region_positions.x, region_positions.y, region_positions.z, marker="o", c=corrected)
            plt.colorbar(scat, cax=ax[1], shrink=0.6).ax.set_ylabel('KS statistcs', rotation=90)

            ax1.set_xlabel('Frontal')
            ax1.set_ylabel('Anteroposterior')
            ax1.set_zlabel('Craniocaudal')
            plt.suptitle("Non-Linearity position")
            plt.show()

        fig, ax = plt.subplots(1,2, gridspec_kw={'width_ratios': [1,1]},figsize=(10,6))
        rnl = 1 - all_regions_MI["GMI"]/all_regions_MI["TMI"]#(all_regions_MI["TMI"]-all_regions_MI["GMI"])/all_regions_MI["sGMI"]#
        num_reg_shown = np.min([10, np.count_nonzero(np.isfinite(corrected))])
        sa = np.argsort(corrected)[::-1]
        last = sa[-2:]
        if np.count_nonzero(np.isnan(corrected))>=2:
            nons = np.argsort(regions_ks_stat)[1::-1]
        else:
            nons = np.array([])
        to_show = np.roll(sa,-np.count_nonzero(np.isnan(corrected)))[:num_reg_shown]
        plt.sca(ax[0])
        plt.boxplot([rnl[i,:] for i in to_show]+[rnl[i,:] for i in last]+[rnl[i,:] for i in nons], showmeans=True)
        plt.xlabel(f"{num_reg_shown} (most) significant regions")
        plt.ylim(plt.ylim())
        plt.vlines([num_reg_shown+0.5, num_reg_shown+2.5], *plt.ylim(), colors = "green", linestyles="dashed", lw=1)
        plt.ylabel("Relative non linearity")
        if region_positions is not None:
            labels = region_positions.loc[np.concatenate([to_show,last,nons])].Labels.to_numpy()
        else:
            labels = np.arange(num_reg_shown+4)+1
        plt.xticks(np.arange(num_reg_shown+2+len(nons))+1, labels, rotation=90)
        plt.xlim(plt.xlim())
        plt.hlines([0,0.1],*plt.xlim(), ["k","r"],["solid","dashed"])

        plt.sca(ax[1])
        top_reg = np.argmax(corrected)
        top_sub = np.argsort(rnl[top_reg,:])[-15:]
        plt.barh(np.arange(15),all_regions_MI["TMI"][top_reg,top_sub],color="darkseagreen", label="Total MI")
        plt.barh(np.arange(15),all_regions_MI["GMI"][top_reg,top_sub], xerr=all_regions_MI["sGMI"][top_reg,top_sub],color="indianred", label="Surrogates MI")
        plt.ylabel(f"15 most nonlinear subjects for most significant region ({region_positions.loc[to_show[0],'Labels'] if region_positions is not None else '-'})")
        plt.xlabel("MI")
        plt.legend(frameon=False,loc=0)
        plt.show()

        plt.scatter(regions_ks_stat[np.isnan(corrected)], siginreg[np.isnan(corrected)], color="gray", label="non sig. KS", alpha=0.6)
        plt.scatter(regions_ks_stat[np.isfinite(corrected)], siginreg[np.isfinite(corrected)], color="blue", label="sig. KS", alpha=0.6)
        plt.xlabel("Region KS statistics")
        plt.ylabel("Sig. non-lin. pairs for region")
        plt.legend(loc=0)
        plt.title(subset_description+subset_na+f" - methods comparison - r: {pearsonr(regions_ks_stat,siginreg)[0]:.3}")
        plt.show()

        zscores = (all_regions_MI["TMI"]-all_regions_MI["GMI"])/all_regions_MI["sGMI"]
        zcorrks = np.array([pearsonr(zscores[:,i], regions_ks_stat) for i in range(subj_num)])
        sorted_p = np.sort(zcorrks[:,1])[::-1]

        distance_from_thresh = sorted_p - 0.05/(subj_num-np.arange(subj_num))
        distance_from_thresh[distance_from_thresh>0]=-np.inf
        which = np.argmax(distance_from_thresh)
        thresh = sorted_p[which] # holm-bonferroni 1-(1-0.05)**(1/pair_num)# šidák 0.05/pair_num # bonferroni
        mask = zcorrks[:,1]<thresh
        signum = np.sum(mask)
        plt.scatter(np.arange(signum), np.sort(zcorrks[mask,0])[::-1])
        plt.scatter(np.arange(signum, subj_num), np.sort(zcorrks[np.logical_not(mask),0])[::-1], color="gray")
        plt.ylabel("Correlation z-score to ks-statistics")
        plt.xlabel("Subject")
        plt.show()
        

In [ ]:
46/144

# fMRI

In [ ]:
pair_num = 4005
subj_num = 245
elec_num = 90
quantiles = [.6,.9]
results_file = "../NonLinearityResults/eso245_aal_{}_90_bin9/patient{:02}_9.npy"
subset_description = "fMRI AAL90 "
subset_identifiers = ["strin"]# ["raw", "mod", "strin"]
subset_names = ["strin"]# ["raw", "mod", "strin"]
output_prefix = "fMRI_AAL90"
region_positions = pd.read_csv("AAL_90regions.csv").loc[slice(None), ["Labels","x","y","z"]]
region_positions["Labels"] = region_positions["Labels"].apply(lambda x: x.split()[1])
nbins = 9
nsteps = loadmat(f"/mnt/DATA/Motion_fMRI/Datasets/eso245/eso245_aal_strin_90.mat")["subj_tcs"].shape[0]
summarise_localised_non_linearity(results_file, subset_description, subset_identifiers, subset_names, output_prefix, pair_num, subj_num, elec_num, region_positions, quantiles, nbins, nsteps)
subset_description = "SHA fMRI AAL90 "
output_prefix = "fMRI_AAL90_sha"
results_file = "../NonLinearityResults/eso245_aal_{}_90_bin9/patient{:02}_9_sha.npy"
summarise_localised_non_linearity(results_file, subset_description, subset_identifiers, subset_names, output_prefix, pair_num, subj_num, elec_num, region_positions, quantiles, nbins, nsteps)

In [ ]:
pair_num = 4005
subj_num = 245
elec_num = 90
for prep in ["raw", "mod", "strin"]:
    ks_stat = np.load(f"../NonLinearityResultsNew/localised/fMRI_AAL90_ks_stat_{prep}.npy")
    ks_p = np.load(f"../NonLinearityResultsNew/localised/fMRI_AAL90_ks_p_{prep}.npy")
    sorted_p = np.sort(ks_p.flatten())[::-1]
    distance_from_thresh = sorted_p - 0.05/(pair_num-np.arange(pair_num))
    distance_from_thresh[distance_from_thresh>0]=-np.inf
    which = np.argmax(distance_from_thresh)

    corrected = np.full(pair_num, np.nan)
    thresh = sorted_p[which] # holm-bonferroni 1-(1-0.05)**(1/pair_num)# šidák 0.05/pair_num # bonferroni
    corrected[ks_p<thresh] = ks_stat[ks_p<thresh]
    mappa = np.zeros([elec_num,elec_num])
    mappa[np.triu_indices(elec_num,1)]=corrected
    mappa+=mappa.T
    siginreg = np.sum(mappa>0, 1)

    Left = siginreg[::2]
    Right = siginreg[1::2]
    print(prep, pearsonr(Left, Right))

# EEG - scalp electrode

In [ ]:
file = zipfile.PyZipFile("../NonLinearityData/EEG_el_so_BLP_20230714.zip")
archive = h5py.File(io.BytesIO(file.read("info.mat")))
zip_file = zipfile.PyZipFile("../NonLinearityData/EEG_el_so_BLP_20230714.zip")
infoMat = h5py.File(io.BytesIO(zip_file.read("info.mat")))
vec= np.where(infoMat["elec_in_mask"])[1]
region_positions = pd.read_csv("electrode_positions.csv").loc[vec, ["Labels","x","y","z"]].reset_index()

pair_num = 18915
subj_num = 50
elec_num = 195
quantiles = [.7,.95]
results_file = "../NonLinearityResultsNew/NEW_EEG_fixedSamples_band{}_bin10/subject{:02}_10.npy"
subset_description = "EEG band "
subset_identifiers = range(1,10)
subset_names = [r"$\delta$",r"$\theta$",r"$\alpha$",r"$\beta_{LOW}$",r"$\beta_{MID}$",r"$\beta_{HIGH}$",r"$\gamma$",r"$\beta_{ALL}$",r"$1-40 Hz$"]
output_prefix = "electrodeEEG"
nbins = 10
nsteps = loadmat(f"/home/raffaelli/NonLinearity/NonLinearityData/EEG_el_so_BLP_NEW/NEW_EEG_fixedSamples_band1.mat")["EEG"].shape[0]
summarise_localised_non_linearity(results_file, subset_description, subset_identifiers, subset_names, output_prefix, pair_num, subj_num, elec_num, region_positions, quantiles, nbins, nsteps)

# EEG - scalp BLP

In [ ]:
file = zipfile.PyZipFile("../NonLinearityData/EEG_el_so_BLP_20230714.zip")
archive = h5py.File(io.BytesIO(file.read("info.mat")))
zip_file = zipfile.PyZipFile("../NonLinearityData/EEG_el_so_BLP_20230714.zip")
infoMat = h5py.File(io.BytesIO(zip_file.read("info.mat")))
vec= np.where(infoMat["elec_in_mask"])[1]
region_positions = pd.read_csv("electrode_positions.csv").loc[vec, ["Labels","x","y","z"]].reset_index()

pair_num = 18915
subj_num = 50
elec_num = 195
quantiles = [.7,.95]
results_file = "../NonLinearityResultsNew/NEW_electrodeBLP_fixedTime_avg2_band{}_bin8/subject{:02}_8.npy"
subset_description = "BLP band "
subset_identifiers = range(1,10)
subset_names = [r"$\delta$",r"$\theta$",r"$\alpha$",r"$\beta_{LOW}$",r"$\beta_{MID}$",r"$\beta_{HIGH}$",r"$\gamma$",r"$\beta_{ALL}$",r"$1-40 Hz$"]
output_prefix = "electrodeBLP"
nbins = 8
nsteps = loadmat(f"/home/raffaelli/NonLinearity/NonLinearityData/EEG_el_so_BLP_NEW/NEW_electrodeBLP_fixedTime_avg2_band1.mat")["EEG"].shape[0]
summarise_localised_non_linearity(results_file, subset_description, subset_identifiers, subset_names, output_prefix, pair_num, subj_num, elec_num, region_positions, quantiles, nbins, nsteps)

# EEG - source BLP

**Doesn't work as I didn't save ALL the values for all the pairs (1e6) for all the bands, averages, and subjects (~4 TB)**

In [ ]:
assert False
region_positions = None

pair_num = 1072380
subj_num = 50
elec_num = 1465
quantiles = [.7,.95]
results_file = "../NonLinearityResultsNew/NEW_sourceBLP_fixedTime_avg2_band{}_bin8/subject{:02}_8.npy"
subset_description = "BLP band "
subset_identifiers = range(1,10)
subset_names = [r"$\delta$",r"$\theta$",r"$\alpha$",r"$\beta_{LOW}$",r"$\beta_{MID}$",r"$\beta_{HIGH}$",r"$\gamma$",r"$\beta_{ALL}$",r"$1-40 Hz$"]
output_prefix = "sourceBLP"
nbins = 8
nsteps = loadmat(f"/home/raffaelli/NonLinearity/NonLinearityData/EEG_el_so_BLP_NEW/NEW_sourceBLP_fixedTime_avg2_band1.mat")["EEG"].shape[0]
summarise_localised_non_linearity(results_file, subset_description, subset_identifiers, subset_names, output_prefix, pair_num, subj_num, elec_num, region_positions, quantiles, nbins, nsteps)

In [ ]:
assert False

In [ ]:
def vecmod (X):
    return np.sqrt(np.sum(X**2))
def dist (X, Y):
    return vecmod(X-Y)

In [ ]:


vertices = reg.loc[slice(None), ["X","Y","Z"]].to_numpy()
T = []
for i in range(90):
    pi = i%2
    x = 30 if pi else -30
    if dist(vertices[i], np.array([x, 0, 15])) < 20:
        continue
    for j in range(i+1, 90):
        pj = j%2
        x = 30 if pj else -30
        if dist(vertices[j], np.array([x, 0, 15])) < 20:
            continue
        for k in range(j+1, 90):
            pk = k%2
            if pi+pj+pk in {1,2}:
                continue
            x = 30 if pk else -30
            if dist(vertices[k], np.array([x, 0, 15])) < 20:
                continue
            d12 = dist(vertices[i], vertices[j])
            if d12>45:
                continue
            d13 = dist(vertices[i], vertices[k])
            if d13>45:
                continue
            d23 = dist(vertices[k], vertices[j])
            if d23>45:
                continue
            if np.max(np.diff(vertices[[i,j,k],0]))>30:
                continue
            T.append([i,j,k])
fig = plt.figure()
ax = fig.add_subplot(111,projection='3d')
ax.plot_trisurf(vertices[:,0], vertices[:,1], vertices[:,2], triangles = T, edgecolor=[[0,0,0]], linewidth=1.0, alpha=0.0, shade=False)

plt.show()

In [ ]:
def read_n_lines(f, nline,ncols = 3, dtype = float, sep = ' '):
    m = np.zeros((nline, ncols), dtype = dtype)
    for i in range(nline):
        line = f.readline()
        try:
            m[i,:] = [dtype(v)  for v in line.split(sep) ]
        except:
            print('error', i, line)
    return m


def read_mesh(filename):
    with open(filename)as f:
        nline =  int(f.readline())
        coords = read_n_lines(f, nline,ncols = 3, dtype = float, sep = ' ')
        nline =  int(f.readline())
        faces = read_n_lines(f, nline,ncols = 3, dtype = int, sep = ' ')-1
    return coords, faces

In [ ]:
vertices, faces = read_mesh("BrainMesh_ICBM152.nv")

fig = plt.figure()
ax = fig.add_subplot(111,projection='3d')
ax.plot_trisurf(subs[:,0], subs[:,1], subs[:,2], triangles = newtri, edgecolor=[[0,0,0]], linewidth=1.0, alpha=0.0, shade=False)
siginreg = np.sum(mappa>0, 1)

linear = reg.iloc[siginreg==0]
mild = reg.iloc[np.logical_and(siginreg>0, siginreg<=3)]
very = reg.iloc[siginreg>3]

ax.scatter(linear.X, linear.Y, linear.Z, marker="o", color="gray")
ax.scatter(mild.X, mild.Y, mild.Z, marker="o", color="orange")
ax.scatter(very.X, very.Y, very.Z, marker="o", color="red")

ax.set_xlabel('X Label')
ax.set_ylabel('Y Label')
ax.set_zlabel('Z Label')
plt.show()

In [ ]:
from scipy.spatial import Delaunay

points = vertices[::50,:]
tri = Delaunay(points)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111,projection='3d')
ax.plot_trisurf(points[:,0], points[:,1], points[:,2], triangles = tri.simplices, edgecolor=[[0,0,0]], linewidth=1.0, alpha=0.0, shade=False)

plt.show()

In [ ]:
subs = vertices[::100,:]
npt = subs.shape[0]
newtri = []
for i in tqdm(range(npt)):
    dists=[dist(subs[i], subs[j]) for j in range(i+1,npt)]
    neigh=np.argsort(dists)[:3]+i+1
    for n1 in range(3):
        for n2 in range(n1+1, 3):
            newtri.append([i, neigh[n1], neigh[n2]])


In [ ]:
pd.read_sql("../../Downloads/pat_26402_2012-12-20.sql")

# Freiburg dataset
## Extraction

These tasks are performed better by the script `../../Downloads/cleaningScript.py`

In [ ]:
from glob import glob
for file_name in glob("../../Downloads/pat_*.sql"):
    headers = {}
    data = {}
    with open(file_name) as file:
        for line in file:
            if line.startswith("INSERT"):
                pieces = line.strip()[:-1].replace("'",'"').split()
                values_ind = pieces.index("VALUES")
                header = " ".join(pieces[3:values_ind])[1:-1]
                row = " ".join(pieces[values_ind+1:])[1:-1]
                if pieces[2] in headers:
                    assert headers[pieces[2]] == header
                    data[pieces[2]].append(row)
                else:
                    headers[pieces[2]] = header
                    data[pieces[2]] = [row]
    for tab in headers:
        with open(f"{file_name[:-4]}_{tab}.csv", "w") as file:
            print(headers[tab], file=file)
            for row in data[tab]:
                print(row, file=file)

In [ ]:
tables = {}
for tab in headers:
    tmp = []
    for file_name in glob("../../Downloads/pat_*.sql"):
        true_file_name = f"{file_name[:-4]}_{tab}.csv"
        try:
            tmp.append(pd.read_csv(true_file_name,  sep=',', quotechar='"', skipinitialspace=True, engine='python'))
        except FileNotFoundError:
            print(true_file_name, " missing.")
    tables[tab]=pd.concat(tmp).reset_index(drop=True)

In [ ]:
for tab in tables:
    display(tables[tab])

In [ ]:
for tab in tables:
    tables[tab].to_csv(f"/home/raffaelli/Downloads/{tab}.csv")

In [ ]:
tables2={}
for tab in tables:
    tables2[tab]=pd.read_csv(f"/home/raffaelli/Downloads/{tab}.csv")

## Reverse engineering

### Relevant links

1. https://www.fieldtriptoolbox.org/faq/coordsys/#details-of-the-analyze-coordinate-system
1. https://eeg.sourceforge.net/ANALYZE75.pdf
1. https://imaging.mrc-cbu.cam.ac.uk/imaging/FormatAnalyze
1. http://www.grahamwideman.com/gw/brain/analyze/formatdoc.htm
1. https://eeg.sourceforge.net/mri_orientation_notes.html
1. https://bids-specification.readthedocs.io/en/stable/appendices/coordinate-systems.html


In [ ]:
%matplotlib widget
import json
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import io, h5py, zipfile
import numpy as np
from tqdm import tqdm
import os
import mienc.support as ms
from mienc import Corrector, NonLinearEstimator
import pandas as pd
import seaborn as sns

In [ ]:
elec_pos = pd.read_csv("../../Downloads/electrode.csv", index_col=0)
elec_pos.dropna(subset=["coord_x","coord_y","coord_z"], inplace=True)
elec_arr = pd.read_csv("../../Downloads/electrode_array.csv", index_col=0).drop(columns=["name","commentary"])
elec = elec_pos.join(elec_arr,on="array")
elec["sid"] = [int(str(a)[:-6]) for a in elec.index.values]
elec["eid"] = [int(str(a)[-6:-3]) for a in elec.index.values]
elec.set_index(["sid","eid"], inplace=True)
elec.dropna(axis=1, how="all", inplace=True)
elec.sort_index()
eeg_focus = pd.read_csv("../../Downloads/eeg_focus.csv", index_col=0).dropna(axis=1)
eeg_focus["sid"] = [int(str(a)[:-6]) for a in eeg_focus.index.values]
eeg_focus["eid"] = [int(str(a)[-6:-3]) for a in eeg_focus.index.values]
eeg_focus.set_index(["sid","eid"], inplace=True)

### Front-back analysis

In [ ]:
eeg_focus["coronal"]=eeg_focus["localisation"].apply(lambda x: x[0])
coronal = eeg_focus.groupby("sid").apply(lambda df: "f" if "f" in df.coronal.unique() else "o" if "o" in df.coronal.unique() else None).dropna()#
frontal_sub = coronal[coronal=="f"].index
occipital_sub = coronal[coronal=="o"].index
coronal

#### Occipital subject

In [ ]:
tmp = elec.sort_index().loc[occipital_sub]
occipital_focus = tmp[tmp["focus_rel"]=="i"].index

In [ ]:
fig, ax = plt.subplots(1,2, gridspec_kw={'width_ratios': [8,0.1]},figsize=(8,6))
ax1 = fig.add_subplot(121,projection='3d')
ax[0].axis("off")
ax[1].axis("off")
for etype, color in zip(['strip', 'grid'], "gb"):#, 'depth', 'surf'rk
    ind = elec[elec["type"]==etype].index
    scat = ax1.scatter(elec.loc[ind, "coord_x"], elec.loc[ind, "coord_y"], elec.loc[ind, "coord_z"], marker="o", color=color, label=etype, s=2)
scat = ax1.scatter(elec.loc[occipital_sub, "coord_x"], elec.loc[occipital_sub, "coord_y"], elec.loc[occipital_sub, "coord_z"], marker="+", color="orange", label="all_1146", s=8)
scat = ax1.scatter(elec.loc[occipital_focus, "coord_x"], elec.loc[occipital_focus, "coord_y"], elec.loc[occipital_focus, "coord_z"], marker="*", color="r", label="SOZ_1146")

ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')
plt.legend()
plt.show()


#### Frontal subjects

In [ ]:
tmp = elec.sort_index().loc[frontal_sub]
frontal_sub_focus = tmp[tmp["focus_rel"]=="i"].index

In [ ]:
fig, ax = plt.subplots(1,2, gridspec_kw={'width_ratios': [8,0.1]},figsize=(8,6))
ax1 = fig.add_subplot(121,projection='3d')
ax[0].axis("off")
ax[1].axis("off")
for etype, color in zip(['strip', 'grid'], "gb"):#, 'depth', 'surf'rk
    ind = elec[elec["type"]==etype].index
    scat = ax1.scatter(elec.loc[ind, "coord_x"], elec.loc[ind, "coord_y"], elec.loc[ind, "coord_z"], marker="o", color=color, label=etype, s=2)
scat = ax1.scatter(elec.loc[frontal_sub, "coord_x"], elec.loc[frontal_sub, "coord_y"], elec.loc[frontal_sub, "coord_z"], marker="+", color="orange", label="all_1146", s=8)
scat = ax1.scatter(elec.loc[frontal_sub_focus, "coord_x"], elec.loc[frontal_sub_focus, "coord_y"], elec.loc[frontal_sub_focus, "coord_z"], marker="*", color="r", label="SOZ_1146")

ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')
plt.legend()
plt.show()

### Side analysis

In [ ]:
eeg_focus["side"]=eeg_focus["localisation"].apply(lambda x: x[-1])
sided = eeg_focus.groupby("sid").apply(lambda df: df.side.unique()[0] if len(df.side.unique())==1 and df.side.unique()[0]!="b" else None).dropna()
right_sub = sided[sided=="r"].index
left_sub = sided[sided=="l"].index
print(f"Only right: {len(right_sub)}, only left: {len(left_sub)}")

#### Show right

In [ ]:
tmp = elec.sort_index().loc[right_sub]
right_sub_focus = tmp[tmp["focus_rel"]=="i"].index

In [ ]:
fig, ax = plt.subplots(1,2, gridspec_kw={'width_ratios': [8,0.1]},figsize=(8,6))
ax1 = fig.add_subplot(121,projection='3d')
ax[0].axis("off")
ax[1].axis("off")
# for etype, color in zip(['strip', 'grid'], "gb"):#, 'depth', 'surf'rk
#     ind = elec[elec["type"]==etype].index
#     scat = ax1.scatter(elec.loc[ind, "coord_x"], elec.loc[ind, "coord_y"], elec.loc[ind, "coord_z"], marker="o", color=color, label=etype, s=2)
scat = ax1.scatter(elec.loc[right_sub, "coord_x"], elec.loc[right_sub, "coord_y"], elec.loc[right_sub, "coord_z"], marker="+", color="orange", label="all_right", s=10)
scat = ax1.scatter(elec.loc[right_sub_focus, "coord_x"], elec.loc[right_sub_focus, "coord_y"], elec.loc[right_sub_focus, "coord_z"], marker="*", color="r", label="SOZ_right")

ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')
plt.legend()
plt.show()

#### Show left

In [ ]:
tmp = elec.sort_index().loc[left_sub]
left_sub_focus = tmp[tmp["focus_rel"]=="i"].index

In [ ]:
fig, ax = plt.subplots(1,2, gridspec_kw={'width_ratios': [8,0.1]},figsize=(8,6))
ax1 = fig.add_subplot(121,projection='3d')
ax[0].axis("off")
ax[1].axis("off")
# for etype, color in zip(['strip', 'grid'], "gb"):#, 'depth', 'surf'rk
#     ind = elec[elec["type"]==etype].index
#     scat = ax1.scatter(elec.loc[ind, "coord_x"], elec.loc[ind, "coord_y"], elec.loc[ind, "coord_z"], marker="o", color=color, label=etype, s=2)
scat = ax1.scatter(elec.loc[left_sub, "coord_x"], elec.loc[left_sub, "coord_y"], elec.loc[left_sub, "coord_z"], marker="+", color="orange", label="all_left", s=8)
scat = ax1.scatter(elec.loc[left_sub_focus, "coord_x"], elec.loc[left_sub_focus, "coord_y"], elec.loc[left_sub_focus, "coord_z"], marker="*", color="r", label="SOZ_left")

ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')
plt.legend()

In [ ]:
import numpy as np

mi = np.power(1.05,1/365)
(7070*mi**26)*mi**39, mi, 7070*0.08/365*7